# 🧠 MemorySparcity — Neuromorphic SNN on N-MNIST
### Full Pipeline: Clone → Preprocess → Train (Sparse SRAM-Backed SNN)

**Before running:** Go to `Runtime > Change runtime type` and set **Hardware accelerator → GPU (T4)**.

---
## ⚠️ Data Setup (One-Time)
Upload both zip files to a **Google Drive** folder, e.g.:
```
My Drive/MemorySparcity/Train.zip   (~1 GB)
My Drive/MemorySparcity/Test.zip    (~162 MB)
```
Then update `GDRIVE_DATA_DIR` in **Cell 3**.

## Step 1: Verify GPU

In [ ]:
import torch
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')
else:
    raise RuntimeError('No GPU detected! Go to Runtime > Change runtime type and select GPU.')

## Step 2: Clone Repository

In [ ]:
import os

REPO_URL = 'https://github.com/SiddheshUttarwar/MemorySparcity.git'
REPO_DIR = '/content/MemorySparcity'

if os.path.exists(REPO_DIR):
    print('Repo already cloned — pulling latest...')
    !git -C {REPO_DIR} pull
else:
    !git clone {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
print(f'Working directory: {os.getcwd()}')
!ls

## Step 3: Mount Google Drive and Copy Data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ✏️ UPDATE this to match your Drive folder
GDRIVE_DATA_DIR = '/content/drive/MyDrive/MemorySparcity'

import shutil

for fname in ['Train.zip', 'Test.zip']:
    src = os.path.join(GDRIVE_DATA_DIR, fname)
    dst = os.path.join(REPO_DIR, fname)
    if not os.path.exists(dst):
        print(f'Copying {fname} from Drive ...')
        shutil.copy2(src, dst)
        print(f'  Done — {os.path.getsize(dst)/1e9:.2f} GB')
    else:
        print(f'{fname} already present, skipping.')

## Step 4: Install Dependencies

In [ ]:
!pip install -q numpy scipy matplotlib

## Step 5: Preprocess N-MNIST Dataset

Reads event `.bin` files directly from the zip archives and produces `preprocessed_data_native/` with `train_*.npz` and `test_*.npz` files.

Each file contains:
- `data`: spike tensor `[T=20, C=2, H=28, W=28]` (bool)
- `digit`: integer class label (0–9)

In [ ]:
import glob

OUT_DIR = os.path.join(REPO_DIR, 'preprocessed_data_native')

# Check if already preprocessed
existing = glob.glob(os.path.join(OUT_DIR, '*.npz'))
if len(existing) > 1000:
    print(f'Preprocessing already done ({len(existing)} files found). Skipping.')
else:
    print('Running preprocessor (multiprocessing, ~2 min) ...')
    !python preprocess_dataset.py

# Verify output
npz_files = glob.glob(os.path.join(OUT_DIR, '*.npz'))
train_files = [f for f in npz_files if os.path.basename(f).startswith('train')]
test_files  = [f for f in npz_files if os.path.basename(f).startswith('test')]
print(f'Train samples: {len(train_files)}')
print(f'Test  samples: {len(test_files)}')

if train_files:
    import numpy as np
    s = np.load(train_files[0])
    print(f'Keys in npz  : {list(s.keys())}')
    print(f'Spike tensor : {s["data"].shape}  dtype={s["data"].dtype}')
    print(f'Label        : {int(s["digit"])}')

## Step 6 (Optional): Fast CNN Baseline

Non-spiking CNN that collapses the time dimension. Should reach **>98% accuracy in ~2 min** — confirms the dataset pipeline is healthy before running the SNN.

In [ ]:
!python train_fast_cnn.py

## Step 7: Train the Sparse Convolutional SNN (CSNN)

This trains a **LeNet-5 CSNN** with:
- **Surrogate Gradient Backprop (STBP)** — Fast Sigmoid, α=10
- **Soft-Reset LIF neurons** — β=0.9, V_th=1.0
- **SRAM-backed weights** — weights are mirrored into `SRAMWeightMemory` blocks after each epoch
- **CosineAnnealing LR** — for smooth convergence
- **20 epochs** — SNNs need more epochs than standard CNNs to converge

Expected behaviour: accuracy climbs steadily from epoch 1 (starts >20%), reaching **>90% by epoch 15–20**.

> If you only want a quick test run, add `--epochs 5`. Full training: `--epochs 20` (default).

In [ ]:
!python train.py --epochs 20 --lr 1e-3 --batch 32

---
## 💾 Save Results to Google Drive

In [ ]:
import glob, shutil

SAVE_DIR = os.path.join(GDRIVE_DATA_DIR, 'results')
os.makedirs(SAVE_DIR, exist_ok=True)

for pattern in ['*.pth', 'plots/*.png', 'plots_compare/*.png']:
    for f in glob.glob(os.path.join(REPO_DIR, pattern)):
        dst = os.path.join(SAVE_DIR, os.path.basename(f))
        shutil.copy2(f, dst)
        print(f'Saved: {dst}')

print('\nAll results saved to Google Drive!')